# Inside the Model: Gradient Boosting

## What does the next tree learn?

**Optional advanced exercise · 35–40 additional minutes**

Trace two stages of squared-error gradient boosting and investigate what changes when the learning rate changes.

**Prerequisite:** Residuals, squared-error fitting and regression trees with constant predictions in their leaves.

### Learning objectives

- calculate the targets passed to successive squared-error boosting trees;
- interpret negative tree outputs as corrections;
- distinguish refitting with shrinkage from shrinking an already fitted ensemble.

Predict and calculate before Python. The optional challenge is additional.


### The experiment

| Input x | Response y |
|--:|--:|
| 0 | 0 |
| 1 | 0 |
| 2 | 2 |
| 3 | 4 |

Use two depth-1 regression trees (stumps), equally weighted observations, squared-error fitting and no subsampling. Candidate thresholds are 0.5, 1.5 and 2.5. Each leaf predicts the mean of its current targets.

For pointwise loss $\ell(y,F)=\tfrac12(y-F)^2$, the negative derivative with respect to $F$ is $y-F$. Thus at stage $m$,

$$r_i^{(m)}=y_i-F_{m-1}(x_i),\qquad
F_m(x)=F_{m-1}(x)+\eta h_m(x).$$

Fit $h_m$ to these residual targets by minimising their squared error. Start with the best constant $F_0$ and initially use $\eta=1$. We are using pointwise negative gradients; differentiating a mean empirical loss as a vector would add a common $1/n$ factor.

This is a specific squared-error gradient-boosting construction. Do not assume that ordinary residuals describe every boosting loss, AdaBoost, or all XGBoost internals.

---


## 1. Predict the first correction

**About 4 minutes · No Python**

Which constant minimises the sum of squared errors? Find $F_0$, then predict the signs of the first tree's targets. Can a useful first tree have negative predictions even though every observed response is nonnegative?

**Your prediction and explanation:** _Write here._


## 2. Fit a tree to the residuals

**About 9 minutes · No Python**

Calculate the four residual targets. For each candidate threshold, obtain the two leaf means and the residual-fit SSE. Choose the smallest SSE.

| Threshold | Left leaf correction | Right leaf correction | Residual-fit SSE |
|--:|--:|--:|--:|
| 0.5 | … | … | … |
| 1.5 | … | … | … |
| 2.5 | … | … | … |

Use the selected stump to calculate $F_1$ at all four inputs, then form the next targets $r^{(2)}=y-F_1$. Distinguish the tree's correction $h_1$ from the accumulated prediction $F_1$.

To keep the focus on changing targets, you may use the following fact for stage 2: **its best threshold is 2.5**. Calculate the two leaf means from your new residuals and update to $F_2$.

| Input | First residual target | First accumulated prediction | Second residual target | Final accumulated prediction |
|--:|--:|--:|--:|--:|
| 0 | … | … | … | … |
| 1 | … | … | … | … |
| 2 | … | … | … | … |
| 3 | … | … | … | … |

**Your calculation and explanation:** _State what the second tree is trying to predict and why it differs from y._


## 3. Halve the learning rate and start again

**About 10 minutes · No Python**

Restore the same initial constant and change **only** $\eta$ from 1 to $1/2$. Refit both stages.

Before calculating, predict which quantities stay the same: $F_0$, the first residual targets, the first stump, $F_1$, the second residual targets, or the second stump's leaf corrections. Explain your choices using the order of operations.

**Your prediction:** _Write here before working through the refit._

### Recompute the next target

Calculate the new $F_1$ and the four stage-2 residuals. In this example the best second threshold remains 2.5; calculate its new leaf corrections and $F_2$.

Compare the result with reusing the **old** two stumps fitted at $\eta=1$, then halving their contributions:

$$F_{\mathrm{reuse}}(x)=F_0+\tfrac12 h_1^{(\eta=1)}(x)+\tfrac12 h_2^{(\eta=1)}(x).$$

The constant is not halved. Evaluate at least one input to decide whether this is the same as a refit with $\eta=1/2$. Explain any difference in terms of the targets used to train the second tree.

**Your refit and comparison:** _Record the target vectors, leaf corrections and one decisive prediction comparison._


---

## 4. Verify the two-stage trace

**About 8 minutes · Start coding only after Sections 1–3**

Complete two expressions in the short function: form the current residuals and obtain the fitted stump's corrections. The supplied code applies the update twice and displays the targets, corrections and accumulated predictions. It also compares both stages with `GradientBoostingRegressor` using the same loss, depth, stage count and learning rate.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

x = np.array([0,1,2,3],dtype=float)
X = x.reshape(-1,1)
y = np.array([0,0,2,4],dtype=float)


In [ ]:
# ADD YOUR CODE HERE: replace the two TODO expressions.
def fit_residual_stump(prediction):
    targets = None  # TODO: current residuals
    if targets is None:
        return None
    stump = DecisionTreeRegressor(max_depth=1, random_state=42)
    stump.fit(X, targets)
    correction = None  # TODO: stump predictions at X
    return stump, correction


Compare the displayed targets and predictions with your tables. The plotted lines show the piecewise-constant fitted functions, not interpolated observations. Both panels have matching axes. The printed library differences should be negligible numerical rounding.


In [ ]:
F0 = np.full(len(y), y.mean())
first_fit = fit_residual_stump(F0)
if first_fit is None or first_fit[1] is None:
    print('Complete the two TODO expressions, then rerun these cells.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
    x_plot = np.linspace(-.3, 3.3, 200)
    X_plot = x_plot.reshape(-1, 1)
    for ax, eta in zip(axes, [1.0, 0.5]):
        tree1, h1 = fit_residual_stump(F0)
        F1 = F0 + eta * h1
        tree2, h2 = fit_residual_stump(F1)
        F2 = F1 + eta * h2
        print(f'\nLearning rate {eta}')
        print('Stage 1 and 2 thresholds:', tree1.tree_.threshold[0], tree2.tree_.threshold[0])
        table = pd.DataFrame([y - F0, h1, F1, y - F1, h2, F2], columns=x,
                    index=['Target 1', 'Correction 1', 'Prediction 1',
                           'Target 2', 'Correction 2', 'Prediction 2'])
        print(table.round(6).to_string())
        model = GradientBoostingRegressor(loss='squared_error', learning_rate=eta,
                    n_estimators=2, max_depth=1, subsample=1.0, random_state=42)
        model.fit(X, y)
        library_F1, library_F2 = model.staged_predict(X)
        print('Stage 1 library difference:', np.max(np.abs(F1 - library_F1)))
        print('Stage 2 library difference:', np.max(np.abs(F2 - library_F2)))
        ax.scatter(x, y, color='black', label='Observed y', zorder=4)
        ax.axhline(y.mean(), color='grey', linestyle='dashed', label='Initial constant')
        F1_plot = y.mean() + eta * tree1.predict(X_plot)
        F2_plot = F1_plot + eta * tree2.predict(X_plot)
        ax.plot(x_plot, F1_plot, color='blue', label='After stage 1')
        ax.plot(x_plot, F2_plot, color='red', label='After stage 2')
        ax.set(title=f'Learning rate = {eta}', xlabel='x', ylim=(-.6, 4.5), xlim=(-.3, 3.3))
        ax.grid(True, alpha=.25)
        ax.legend(loc='upper left', fontsize=8)
        if eta == 1.0:
            reused = F0 + 0.5 * h1 + 0.5 * h2
    print('\nReuse old trees, halve corrections:', np.round(reused, 6))
    print('Refit with learning rate 0.5:      ', np.round(F2, 6))
    axes[0].set_ylabel('Accumulated prediction F')
    plt.tight_layout()
    plt.show()


---

## 5. Explain the sequence

**About 5 minutes**

1. Why is a negative leaf correction useful here? Which quantity would it be incorrect to interpret as a direct prediction of y?
2. Why is the first tree unchanged by the learning-rate perturbation, but the second tree's leaf values change?
3. Why does this two-stage training comparison not establish which learning rate generalises better? What would change in the target calculation if the pointwise loss changed?

**Your interpretation:** _Answer in two or three sentences per question._

## Optional Master's challenge

**Allow a further 5–10 minutes**

Keep the four inputs and two-stage construction, but choose responses for which changing the learning rate from 1 to $1/2$ changes the **second stump's split**, not only its leaf values. Give your response vector and compare the candidate residual-fit SSEs at stage 2. Avoid a best-split tie.

**Your construction and evidence:** _Write here._

---

## Takeaway

Complete: **“The next boosting tree learns …, so changing an earlier update …”** Refer to the two stage-2 target vectors.
